# 05_AudioSR.ipynb
### Super-resolución / Bandwidth Extension (BWE) — TFG Restauración interactiva de señales de audio degradadas

Modelo: **AudioSR** (Liu et al., 2023) — modelo de difusión (DDIM) que realiza super-resolución de audio, cualquier entrada → salida a 48kHz.

Repo: `haoheliu/versatile_audio_super_resolution`

Patrón del notebook (igual que en los anteriores):
1. Instalación de dependencias específicas
2. Imports y utils compartidos
3. Carga del modelo
4. Inferencia sobre audio de test
5. Métricas (bloque empírico: no-intrusivas)
6. Baseline clásico (no-IA) de referencia
7. Liberar memoria GPU


## 1. Instalación de dependencias

In [1]:
# ============================================================
# 05_AudioSR.ipynb — Instalación
# ============================================================
# audiosr==0.0.7 declara numpy<=1.23.5 como dependencia, pero esa
# version de numpy no tiene wheel para Python 3.12 (el que usa
# Colab actualmente) y falla al compilar desde source.
# Solucion: instalar sin resolver dependencias (--no-deps), ya que
# torch/torchaudio/torchvision/gradio/einops/etc. ya estan satisfechos
# por el entorno base de Colab.

!pip install audiosr==0.0.7 --no-deps -q
!pip install unidecode phonemizer ftfy torchlibrosa -q
!pip install speechmos pesq pystoi -q
!pip install onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 36.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.8/103.8 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.5/69.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.7/216.7 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 36.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
audiosr 0.0.7 requires progressbar, which is not installed.
audiosr 0.0.7 requires librosa==0.9.2, but you have librosa 0.11.0 which is incompatible.
audiosr 0.0.7 requires numpy<=1.23.5, 

In [2]:
!pip show audiosr

Name: audiosr
Version: 0.0.7
Summary: This package is written for text-to-audio/music generation.
Home-page: https://github.com/haoheliu/audiosr
Author: Haohe Liu
Author-email: haoheliu@gmail.com
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: chardet, einops, ftfy, gradio, huggingface-hub, librosa, numpy, pandas, phonemizer, progressbar, pyyaml, scipy, soundfile, timm, torch, torchaudio, torchlibrosa, torchvision, tqdm, transformers, unidecode
Required-by: 


In [3]:
# Dependencias transitivas de audiosr que --no-deps se salto.
# Ninguna de estas toca numpy/torch, asi que son seguras de instalar sueltas.
!pip install unidecode phonemizer torchlibrosa ftfy timm -q

**⚠️ Aviso importante antes de seguir:** tras ejecutar el `pip install`, comprueba si Colab pide reiniciar el entorno (mensaje típico de "RESTART RUNTIME" o conflicto de versión de numpy/transformers). Si aparece, reinicia el kernel **antes** de ejecutar la celda de imports.

In [1]:
!grep "^def " /content/drive/MyDrive/Proyecto_Audio/utils/audio_utils_baselines_clasicos.py

grep: /content/drive/MyDrive/Proyecto_Audio/utils/audio_utils_baselines_clasicos.py: No such file or directory


## 2. Imports y utils compartidos

In [2]:
import sys, os, time
sys.path.append('/content/drive/MyDrive/Proyecto_Audio/utils')

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import torch
import soundfile as sf
import numpy as np

from audiosr import build_model, super_resolution

from audio_utils_funcionescomunes import cargar_audio, guardar_audio
from audio_utils_metricas_no_intrusivas import calcular_dnsmos
from audio_utils_metricas_ref import calcular_pesq, calcular_stoi, calcular_si_sdr, calcular_lsd
from audio_utils_baselines_clasicos import baseline_bwe_interpolacion_spline   # interpolacion clasica (p.ej. spline/SBR)
from audio_utils_memoria_GPU import liberar_memoria_gpu

BASE_DIR = '/content/drive/MyDrive/Proyecto_Audio'
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")

Mounted at /content/drive


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Dispositivo: cuda


## 3. Carga del modelo

In [3]:
# model_name: "basic" (general) o "speech" (optimizado para voz)
# Usamos "basic" como modelo principal por versatilidad ante distintos
# tipos de senal (coherente con el resto del TFG, que no se limita a voz)

audiosr_model = build_model(model_name="basic", device=device)
print("Modelo AudioSR cargado correctamente.")

Loading AudioSR: basic
Loading model on cuda


pytorch_model.bin:   0%|          | 0.00/6.18G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torchaudio/transforms/_transforms.py:590: UserWarning: Argument 'onesided' has been deprecated and has no influence on the behavior of this module.
  warnings.warn(


DiffusionWrapper has 258.20 M params.


/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Modelo AudioSR cargado correctamente.


Aviso: sin nombre_variable, solo se libera la referencia local a esta función. Si el modelo sigue asignado a una variable en el notebook (ej. modelo_dfn), la GPU no se liberará del todo. Llama a esta función como liberar_memoria_gpu(modelo_dfn, 'modelo_dfn') para liberarla de verdad.
Memoria GPU liberada. Uso actual: 7.03 GB
Memoria liberada


## 4. Inferencia sobre audio de test

In [6]:
import librosa
import soundfile as sf
import numpy as np
import tempfile, os, time

CHUNK_SECONDS = 5.0     # margen bajo el limite de 5.12s recomendado
OVERLAP_SECONDS = 0.1   # solape pequeño para crossfade y evitar clicks
SR_OUT = 48000
SILENCE_RMS_THRESHOLD = 1e-4  # ajustable si detectas falsos positivos/negativos

# --- Seleccion del audio de entrada ---
carpeta_audio = '/content/drive/MyDrive/Proyecto_Audio/audio_samples'
print(f"Archivos disponibles en {carpeta_audio}:\n")
for f in os.listdir(carpeta_audio):
    print(f" - {f}")

nombre_archivo = input("\nIntroduce el nombre exacto del archivo a usar (con extensión): ")
input_path = os.path.join(carpeta_audio, nombre_archivo)

if not os.path.isfile(input_path):
    raise FileNotFoundError(f"No se encontró el archivo: {input_path}")

print(f"\nUsando como input: {input_path}")

output_path = f'{BASE_DIR}/outputs/audiosr_output.wav'

# --- Carga del audio (forzado a mono) ---
# AudioSR devuelve siempre salida en mono independientemente del input,
# asi que para evitar incompatibilidades de forma entre chunks procesados
# por el modelo y chunks silenciosos (resampleados aparte), se baja todo
# a mono desde el principio. Coherente con el tratamiento ya aplicado
# para DNSMOS en notebooks anteriores.
audio, sr_in = librosa.load(input_path, sr=None, mono=True)
audio = audio[np.newaxis, :]  # forzar forma (1, muestras) para mantener el mismo pipeline

n_channels, n_samples = audio.shape
chunk_len_in = int(CHUNK_SECONDS * sr_in)
overlap_len_in = int(OVERLAP_SECONDS * sr_in)

# Definir los puntos de inicio de cada chunk (con solape)
starts = list(range(0, n_samples, chunk_len_in - overlap_len_in))

chunks_out = []
t0 = time.time()

with tempfile.TemporaryDirectory() as tmpdir:
    for i, start in enumerate(starts):
        end = min(start + chunk_len_in, n_samples)
        chunk_audio = audio[:, start:end]

        rms = np.sqrt(np.mean(chunk_audio ** 2))

        if rms < SILENCE_RMS_THRESHOLD:
            print(f"Chunk {i+1}/{len(starts)} silencioso (RMS={rms:.2e}) — se resamplea sin pasar por AudioSR")
            chunk_out = librosa.resample(chunk_audio, orig_sr=sr_in, target_sr=SR_OUT)
            chunks_out.append(chunk_out)
            continue

        chunk_path = os.path.join(tmpdir, f"chunk_{i}.wav")
        sf.write(chunk_path, chunk_audio.T, sr_in)

        print(f"Procesando chunk {i+1}/{len(starts)} ({start/sr_in:.1f}s - {end/sr_in:.1f}s)...")

        try:
            waveform = super_resolution(
                audiosr_model,
                chunk_path,
                seed=42,
                guidance_scale=3.5,
                ddim_steps=50,
                latent_t_per_second=12.8
            )
            chunk_out = waveform[0]  # (1, muestras) a 48kHz
        except ValueError as e:
            print(f"  -> Fallo en AudioSR para este chunk ({e}); se resamplea como fallback")
            chunk_out = librosa.resample(chunk_audio, orig_sr=sr_in, target_sr=SR_OUT)

        # Asegurar forma (1, N) consistente en todos los casos
        if chunk_out.ndim == 1:
            chunk_out = chunk_out[np.newaxis, :]

        chunks_out.append(chunk_out)
        torch.cuda.empty_cache()

print(f"Inferencia por chunks completada en {time.time()-t0:.1f}s")

# --- Recomposicion con crossfade lineal en la zona de solape ---
overlap_len_out = int(OVERLAP_SECONDS * SR_OUT)
final_audio = chunks_out[0]

for chunk in chunks_out[1:]:
    if overlap_len_out > 0 and final_audio.shape[1] >= overlap_len_out and chunk.shape[1] >= overlap_len_out:
        fade_out = np.linspace(1, 0, overlap_len_out)
        fade_in = np.linspace(0, 1, overlap_len_out)

        final_tail = final_audio[:, -overlap_len_out:] * fade_out
        chunk_head = chunk[:, :overlap_len_out] * fade_in
        crossfaded = final_tail + chunk_head

        final_audio = np.concatenate([
            final_audio[:, :-overlap_len_out],
            crossfaded,
            chunk[:, overlap_len_out:]
        ], axis=1)
    else:
        final_audio = np.concatenate([final_audio, chunk], axis=1)

# Guardar resultado final
out_wav = (final_audio * 32767).astype(np.int16).T
sf.write(output_path, out_wav, samplerate=SR_OUT)
print(f"Guardado en: {output_path} — duracion final: {final_audio.shape[1]/SR_OUT:.1f}s")

Archivos disponibles en /content/drive/MyDrive/Proyecto_Audio/audio_samples:

 - AUDIO_REVERB_ALBIOL_TFG.wav

Introduce el nombre exacto del archivo a usar (con extensión): AUDIO_REVERB_ALBIOL_TFG.wav

Usando como input: /content/drive/MyDrive/Proyecto_Audio/audio_samples/AUDIO_REVERB_ALBIOL_TFG.wav
Chunk 1/18 silencioso (RMS=0.00e+00) — se resamplea sin pasar por AudioSR
Procesando chunk 2/18 (4.9s - 9.9s)...
Running DDIM Sampling with 50 timesteps


DDIM Sampler: 100%|██████████| 50/50 [00:09<00:00,  5.34it/s]


Procesando chunk 3/18 (9.8s - 14.8s)...
Running DDIM Sampling with 50 timesteps


DDIM Sampler: 100%|██████████| 50/50 [00:09<00:00,  5.27it/s]


Procesando chunk 4/18 (14.7s - 19.7s)...
Running DDIM Sampling with 50 timesteps


...

Procesando chunk 17/18 (78.4s - 83.4s)...
Running DDIM Sampling with 50 timesteps


DDIM Sampler: 100%|██████████| 50/50 [00:08<00:00,  5.59it/s]


Procesando chunk 18/18 (83.3s - 88.0s)...
Running DDIM Sampling with 50 timesteps


DDIM Sampler: 100%|██████████| 50/50 [00:09<00:00,  5.25it/s]


Inferencia por chunks completada en 195.0s
Guardado en: /content/drive/MyDrive/Proyecto_Audio/outputs/audiosr_output.wav — duracion final: 90.3s


In [8]:
from IPython.display import Audio, display

print("🔊 Audio original (antes de AudioSR):")
display(Audio(input_path))

print("🔊 Audio procesado (después de AudioSR):")
display(Audio(output_path))

Output hidden; open in https://colab.research.google.com to view.

**Nota técnica:** la salida es *siempre* 48kHz, sea cual sea el sample rate de entrada. Si más adelante quieres calcular LSD o comparar espectrogramas frente al original, tendrás que igualar sample rates (resamplear una de las dos señales) antes de la comparación.

## 5. Métricas (bloque empírico: no-intrusivas)

In [10]:
# DNSMOS (auto-resamplea a 16kHz internamente, segun el fix ya aplicado)
audio_out, sr_out = librosa.load(output_path, sr=None, mono=True)
dnsmos_output = calcular_dnsmos(audio_out, sr_out)
print(f"DNSMOS (AudioSR): {dnsmos_output}")

# Si tuvieras referencia limpia (bloque bibliografico con datasets como VCTK):
# lsd_score = calcular_lsd(referencia_path, output_path)

DNSMOS (AudioSR): {'ovrl_mos': np.float64(1.4237323956699606), 'sig_mos': np.float64(1.7587683286796865), 'bak_mos': np.float64(1.9432202738007818), 'p808_mos': np.float32(2.2272127)}


In [9]:
import inspect
print(inspect.signature(calcular_dnsmos))

(audio, sr)


In [18]:
import inspect
print(inspect.signature(baseline_bwe_interpolacion_spline))

(audio, sr_origen, sr_destino)


## 6. Baseline clásico (no-IA) de referencia

In [19]:
baseline_path = f'{BASE_DIR}/outputs/baseline_sr_output.wav'

# Cargar el audio de entrada (mismo tratamiento mono que usamos para AudioSR)
audio_in_baseline, sr_in_baseline = librosa.load(input_path, sr=None, mono=True)

# Ejecutar el baseline clasico de BWE
audio_baseline_out = baseline_bwe_interpolacion_spline(audio_in_baseline, sr_in_baseline, SR_OUT)

# Guardar el resultado
sf.write(baseline_path, audio_baseline_out, samplerate=SR_OUT)

# Calcular DNSMOS sobre el resultado
dnsmos_baseline = calcular_dnsmos(audio_baseline_out, SR_OUT)
print(f"DNSMOS (baseline clasico BWE): {dnsmos_baseline}")

DNSMOS (baseline clasico BWE): {'ovrl_mos': np.float64(1.3853449583283746), 'sig_mos': np.float64(1.5779829548253497), 'bak_mos': np.float64(1.790050888915689), 'p808_mos': np.float32(2.426121)}


In [15]:
import audio_utils_baselines_clasicos as bc
print([f for f in dir(bc) if not f.startswith('_')])

['CubicSpline', 'baseline_bwe_interpolacion_spline', 'baseline_declipping_interpolacion_cubica', 'baseline_denoising_spectral_gating', 'baseline_dereverb_filtro_paso_alto', 'baseline_separacion_hpss', 'librosa', 'np', 'signal']


## 7. Liberar memoria GPU

In [20]:
liberar_memoria_gpu(audiosr_model)
torch.cuda.empty_cache()
print("Memoria GPU liberada.")

Aviso: sin nombre_variable, solo se libera la referencia local a esta función. Si el modelo sigue asignado a una variable en el notebook (ej. modelo_dfn), la GPU no se liberará del todo. Llama a esta función como liberar_memoria_gpu(modelo_dfn, 'modelo_dfn') para liberarla de verdad.
Memoria GPU liberada. Uso actual: 0.01 GB
Memoria GPU liberada.


---
### Notas / posibles problemas a vigilar

1. **Conflicto numpy/transformers** al instalar — mismo patrón que DeepFilterNet, puede pedir reinicio de runtime.
2. **Tiempos de inferencia largos** en T4 con `ddim_steps=50`. Para iterar rápido, prueba temporalmente con `ddim_steps=25` y sube a 50 para la versión final de resultados.
3. **Degradación de calidad si el audio de test no es un low-pass "limpio"** (p.ej. viene de compresión MP3): el propio repo de AudioSR advierte que el modelo espera patrones de corte tipo low-pass, no artefactos de compresión. Puede ser un punto a mencionar en limitaciones.
4. Si usas la grabación estéreo del tutor (bloque empírico), aplica el mismo tratamiento estéreo→mono ya usado en `calcular_dnsmos` para los notebooks anteriores.
